# Supplementary Fig. 2a: power spectral densities of the raw simulated and in vivo traces

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel a of Supplementary Fig. 2: power spectral density of the raw voltage traces of the in vivo recordings and of the simulated recordings, with power laws fitted to the low frequencies.

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import spikeinterface as si
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# set project path
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias-paper/"
os.chdir(PROJ_PATH)

# custom package
import matplotlib
from sklearn.linear_model import LinearRegression
from matplotlib.lines import Line2D
import re

# paths of the PSDs
PSD_PATHS = {
    "H": ["dataset/01_intermediate/psds/psd_raw_horvath_probe1.npy", "dataset/01_intermediate/psds/psd_raw_horvath_probe2.npy", "dataset/01_intermediate/psds/psd_raw_horvath_probe3.npy"],
    "NS": ["dataset/01_intermediate/psds/psd_raw_npx_spont.npy"],
    "E": ["dataset/01_intermediate/psds/psd_raw_npx_evoked.npy"],
    "DS": ["dataset/01_intermediate/psds/psd_raw_dense_probe1.npy", "dataset/01_intermediate/psds/psd_raw_dense_probe2.npy", "dataset/01_intermediate/psds/psd_raw_dense_probe3.npy"],
    "M": ["dataset/01_intermediate/psds/psd_raw_marques_smith.npy"],
}
LABELS = {"M": "M : Marques-Smith", "NS": "NS : Biophy. spont.", "E": "E: Biophy. evoked", "H": "H: Horvath", "DS": "DS: Biophy. spont"}
COLORS = {"M": (0.6, 0.6, 0.6), "NS": (0.9, 0.14, 0.15), "E": (1, 0.49, 0), "H": (0.0, 0.0, 0.0), "DS": np.array([26, 152, 80]) / 255}
LFP_BAND = (0, 90)      # Hz, frequency band of the power-law fit

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 1.1
plt.rcParams["axes.linewidth"] = 1
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}# paths
DATA_PATH = "dataset/01_intermediate/supplementary/lfp_validation/lfp_infinitesimal_vs_finite.csv"

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 1.1
plt.rcParams["axes.linewidth"] = 1
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())     # the repository
from src.nodes import figure_style
figure_style.apply()

## Load the power spectral densities

In [2]:
%%time

psd = {}
for experiment, paths in PSD_PATHS.items():
    parts = [np.load(p, allow_pickle=True).item() for p in paths]
    assert all(p["freq"][1] == 1.0 for p in parts), "psd resolution should be 1 Hz"
    psd[experiment] = {"power": np.vstack([p["power"] for p in parts]), "freq": parts[0]["freq"]}
    print(f"{experiment:3s} sites: {psd[experiment]['power'].shape[0]:4d} | frequencies: {len(psd[experiment]['freq'])} (up to {psd[experiment]['freq'][-1]:.0f} Hz)")

H   sites:  272 | frequencies: 10001 (up to 10000 Hz)
NS  sites:  202 | frequencies: 20001 (up to 20000 Hz)
E   sites:  202 | frequencies: 10001 (up to 10000 Hz)
DS  sites:  297 | frequencies: 10001 (up to 10000 Hz)
M   sites:  226 | frequencies: 15001 (up to 15000 Hz)
CPU times: user 3.79 ms, sys: 26 ms, total: 29.8 ms
Wall time: 29.8 ms


## Mean log power and power-law fit of each experiment

In [3]:
%%time

points, fits = [], []
for experiment, p in psd.items():
    freq, power = p["freq"], p["power"]
    keep = (freq > 0) & np.isfinite(np.log10(freq))
    log_freq, log_power = np.log10(freq[keep]), np.log10(power[:, keep])
    points.append(pd.DataFrame({"experiment": LABELS[experiment], "frequency_Hz": freq[keep], "log10_power_mean_over_sites": log_power.mean(axis=0)}))

    # power law of each site in the 0-90 Hz band
    band = (freq[keep] > LFP_BAND[0]) & (freq[keep] <= LFP_BAND[1])
    coefs, intercepts = [], []
    for site in range(power.shape[0]):
        reg = LinearRegression().fit(log_freq[band].reshape(-1, 1), log_power[site, band])
        coefs.append(reg.coef_[0]); intercepts.append(reg.intercept_)
    fits.append(dict(experiment=LABELS[experiment], n_sites=power.shape[0], slope_mean=np.mean(coefs), slope_std=np.std(coefs), intercept_mean=np.mean(intercepts),
                     band_low_Hz=float(freq[keep][band].min()), band_high_Hz=float(freq[keep][band].max())))
df_points, df_fits = pd.concat(points, ignore_index=True), pd.DataFrame(fits)
display(df_fits.round(3))

<timed exec>:4: RuntimeWarning: divide by zero encountered in log10
<timed exec>:4: RuntimeWarning: divide by zero encountered in log10
<timed exec>:4: RuntimeWarning: divide by zero encountered in log10
<timed exec>:4: RuntimeWarning: divide by zero encountered in log10


<timed exec>:4: RuntimeWarning: divide by zero encountered in log10


,experiment,n_sites,slope_mean,slope_std,intercept_mean,band_low_Hz,band_high_Hz
0,H: Horvath,272,-2.272,0.177,5.162,1.0,90.0
1,NS : Biophy. spont.,202,-0.315,0.033,-1.308,1.0,90.0
2,E: Biophy. evoked,202,-0.584,0.241,-0.335,1.0,90.0
3,DS: Biophy. spont,297,-0.301,0.048,-0.613,1.0,90.0
4,M : Marques-Smith,226,-0.606,0.144,-0.972,1.0,90.0


CPU times: user 302 ms, sys: 9.85 ms, total: 312 ms
Wall time: 311 ms


## Plot

In [ ]:
from src.nodes.validation.psd_plots import plot_psd, downsample_for_display

def plot_source_data(df_points: pd.DataFrame, df_fits: pd.DataFrame):
    return plot_psd(df_points, df_fits)

## Plot and save the figure and the source data

In [ ]:
%%time

df_points = downsample_for_display(df_points, ['experiment'], "log10_power_mean_over_sites")

# plot
fig = plot_source_data(df_points, df_fits)

# save figure
os.makedirs("figures/source_data/supp_fig2a", exist_ok=True)
plt.savefig("figures/source_data/supp_fig2a/supp_fig2a.svg", transparent=True)   # the .pdf is written next to the .svg

# save source data
with pd.ExcelWriter("figures/source_data/supp_fig2a/supp_fig2a.xlsx") as writer:
    df_points.to_excel(writer, sheet_name="psd_mean_over_sites", index=False)
    df_fits.to_excel(writer, sheet_name="power_law_fits", index=False)

## Check: replot from the saved source data

In [ ]:
%%time

df_points = pd.read_excel("figures/source_data/supp_fig2a/supp_fig2a.xlsx", sheet_name="psd_mean_over_sites")
df_fits = pd.read_excel("figures/source_data/supp_fig2a/supp_fig2a.xlsx", sheet_name="power_law_fits")
fig = plot_source_data(df_points, df_fits)